In [1]:
# ==================================================
# CLEAN OGS MINIMAL LIQUID FLOW MODEL
# Dom's lithium project, first OGS test
# ==================================================

import os
import shutil
import subprocess
from pathlib import Path

In [2]:
# --------------------------------------------------
# 1. Define folders
# --------------------------------------------------

base_dir = Path("OGS_files")
input_dir = base_dir / "input"
output_dir = base_dir / "output"

input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print("Input folder:", input_dir)
print("Output folder:", output_dir)


Input folder: OGS_files\input
Output folder: OGS_files\output


In [4]:
# --------------------------------------------------
# 2. Copy mesh into OGS input folder
# --------------------------------------------------

source_mesh = Path(r"Jupyter notebooks/mesh5.vtu")
target_mesh = input_dir / "mesh5.vtu"

shutil.copy(source_mesh, target_mesh)

print("Mesh copied to:", target_mesh)

Mesh copied to: OGS_files\input\mesh5.vtu


In [5]:
# --------------------------------------------------
# 3. Create minimal geometry file
# --------------------------------------------------

gml_content = """<?xml version="1.0" encoding="ISO-8859-1"?>
<OpenGeoSysGLI>
    <name>simple_geometry</name>
    <points>
    </points>
    <polylines>
    </polylines>
</OpenGeoSysGLI>
"""

gml_path = input_dir / "simple_geometry.gml"
gml_path.write_text(gml_content)

print("Geometry file created:", gml_path)

Geometry file created: OGS_files\input\simple_geometry.gml


In [11]:
# --------------------------------------------------
# 4. Create complete minimal OGS project file
# --------------------------------------------------

prj_content = """<?xml version="1.0" encoding="ISO-8859-1"?>
<OpenGeoSysProject>

    <mesh>mesh5.vtu</mesh>

    <processes>
        <process>
            <name>LiquidFlow</name>
            <type>LIQUID_FLOW</type>
            <integration_order>2</integration_order>
            <specific_body_force>0 0 0</specific_body_force>
            <process_variables>
                <process_variable>pressure</process_variable>
            </process_variables>
        </process>
    </processes>

    <time_loop>
        <processes>
            <process ref="LiquidFlow">
                <nonlinear_solver>basic_picard</nonlinear_solver>
                <time_discretization>
                    <type>BackwardEuler</type>
                </time_discretization>
                <convergence_criterion>
                    <type>DeltaX</type>
                    <norm_type>NORM2</norm_type>
                    <abstol>1e-12</abstol>
                </convergence_criterion>
                <time_stepping>
                    <type>FixedTimeStepping</type>
                    <t_initial>0</t_initial>
                    <t_end>10</t_end>
                    <timesteps>
                        <pair>
                            <repeat>10</repeat>
                            <delta_t>1</delta_t>
                        </pair>
                    </timesteps>
                </time_stepping>
            </process>
        </processes>

        <output>
            <type>VTK</type>
            <prefix>result</prefix>
            <timesteps>
                <pair>
                    <repeat>10</repeat>
                    <each_steps>1</each_steps>
                </pair>
            </timesteps>
            <variables>
                <variable>pressure</variable>
            </variables>
        </output>
    </time_loop>

    <media>
        <medium id="0">
            <phases>
                <phase>
                    <type>AqueousLiquid</type>
                    <properties>
                        <property>
                            <name>viscosity</name>
                            <type>Constant</type>
                            <value>1e-3</value>
                        </property>
                        <property>
                            <name>density</name>
                            <type>Constant</type>
                            <value>1000</value>
                        </property>
                    </properties>
                </phase>
            </phases>

            <properties>
                <property>
                    <name>reference_temperature</name>
                    <type>Constant</type>
                    <value>293.15</value>
                </property>
                <property>
                    <name>permeability</name>
                    <type>Constant</type>
                    <value>1e-12</value>
                </property>
                <property>
                    <name>porosity</name>
                    <type>Constant</type>
                    <value>0.2</value>
                </property>
                <property>
                    <name>storage</name>
                    <type>Constant</type>
                    <value>1e-10</value>
                </property>
            </properties>
        </medium>
    </media>

    <parameters>
        <parameter>
            <name>pressure_ic</name>
            <type>Constant</type>
            <value>0</value>
        </parameter>
    </parameters>

    <process_variables>
        <process_variable>
            <name>pressure</name>
            <components>1</components>
            <order>1</order>
            <initial_condition>pressure_ic</initial_condition>
            <boundary_conditions>
            </boundary_conditions>
        </process_variable>
    </process_variables>

    <nonlinear_solvers>
        <nonlinear_solver>
            <name>basic_picard</name>
            <type>Picard</type>
            <max_iter>50</max_iter>
            <linear_solver>linear_solver</linear_solver>
        </nonlinear_solver>
    </nonlinear_solvers>

    <linear_solvers>
        <linear_solver>
            <name>linear_solver</name>
            <eigen>
                <solver_type>CG</solver_type>
                <precon_type>DIAGONAL</precon_type>
                <max_iteration_step>10000</max_iteration_step>
                <error_tolerance>1e-12</error_tolerance>
            </eigen>
        </linear_solver>
    </linear_solvers>

</OpenGeoSysProject>
"""

prj_path = input_dir / "simple_flow.prj"
prj_path.write_text(prj_content)

print("Project file created:", prj_path)


Project file created: OGS_files\input\simple_flow.prj


In [7]:
# --------------------------------------------------
# 5. Run OGS
# --------------------------------------------------

result = subprocess.run(
    ["ogs", "simple_flow.prj", "-o", "../output"],
    cwd=input_dir,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

info: OGS started on 2026-04-28 09:39:28-0600 in serial mode.
info: This is OpenGeoSys-6 version 6.5.7-224-g3022bae09f. Log version: 2, Log level: info.
info: Eigen use 1 threads
info: Reading project file simple_flow.prj.
info: readRasters ...
info: readRasters done
info: ConstantParameter: pressure_ic
info: No source terms for process variable 'pressure' found.
info: Initialize processes.
info: Time step #0 started. Time: 0. Step size: 0.
info: [time] Output of timestep 0 took 0.700949 s.
info: Time step #0 took 0.833994 s.
info: Solve processes.
info: Time step #1 started. Time: 1. Step size: 1.
info: Solving process #0 started.
info: Iteration #1 started.
info: [time] Assembly took 0.812161 s.
info: [time] Applying Dirichlet BCs took 0.0164375 s.
info: ------------------------------------------------------------------
info: *** Eigen solver compute()
info: -> compute with Eigen iterative linear solver CG (precon DIAGONAL)
info: ------------------------------------------------------

In [8]:
# --------------------------------------------------
# 6. Check output files
# --------------------------------------------------

print("Output files:")
print(os.listdir(output_dir))

Output files:
['result.pvd', 'result_ts_0_t_0.000000.vtu', 'result_ts_10_t_10.000000.vtu', 'result_ts_1_t_1.000000.vtu', 'result_ts_2_t_2.000000.vtu', 'result_ts_3_t_3.000000.vtu', 'result_ts_4_t_4.000000.vtu', 'result_ts_5_t_5.000000.vtu', 'result_ts_6_t_6.000000.vtu', 'result_ts_7_t_7.000000.vtu', 'result_ts_8_t_8.000000.vtu', 'result_ts_9_t_9.000000.vtu']


In [9]:
import pyvista as pv

mesh = pv.read("OGS_files/output/result_ts_10_t_10.000000.vtu")

print(mesh)
print("Available arrays:")
print(mesh.array_names)

UnstructuredGrid (0x2da4fce9480)
  N Cells:    166004
  N Points:   27927
  X Bounds:   0.000e+00, 1.000e+01
  Y Bounds:   0.000e+00, 1.000e+01
  Z Bounds:   0.000e+00, 5.000e+00
  N Arrays:   2
Available arrays:
['OGS_VERSION', 'pressure']


In [10]:
from pathlib import Path
import os
import pyvista as pv

base_dir = Path("OGS_files")
input_dir = base_dir / "input"
output_dir = base_dir / "output"

print("Current folder:")
print(os.getcwd())

print("\nInput files:")
print(os.listdir(input_dir))

print("\nOutput files:")
print(os.listdir(output_dir))

print("\nPRJ exists:")
print((input_dir / "simple_flow.prj").exists())

print("\nMesh copy exists:")
print((input_dir / "mesh5.vtu").exists())

final_result = output_dir / "result_ts_10_t_10.000000.vtu"
print("\nFinal result exists:")
print(final_result.exists())

mesh = pv.read(final_result)
print("\nAvailable arrays:")
print(mesh.array_names)

print("\nMesh size:")
print("Cells:", mesh.n_cells)
print("Points:", mesh.n_points)

Current folder:
E:\ADATA\LITHIUM\OGS

Input files:
['mesh5.vtu', 'simple_flow.prj', 'simple_geometry.gml']

Output files:
['result.pvd', 'result_ts_0_t_0.000000.vtu', 'result_ts_10_t_10.000000.vtu', 'result_ts_1_t_1.000000.vtu', 'result_ts_2_t_2.000000.vtu', 'result_ts_3_t_3.000000.vtu', 'result_ts_4_t_4.000000.vtu', 'result_ts_5_t_5.000000.vtu', 'result_ts_6_t_6.000000.vtu', 'result_ts_7_t_7.000000.vtu', 'result_ts_8_t_8.000000.vtu', 'result_ts_9_t_9.000000.vtu']

PRJ exists:
True

Mesh copy exists:
True

Final result exists:
True

Available arrays:
['OGS_VERSION', 'pressure']

Mesh size:
Cells: 166004
Points: 27927
